In [1]:
import clickhouse_connect

from clickhouse_connect.driver.tools import insert_file
import pyarrow
import pandas as pd


In [2]:
client = clickhouse_connect.get_client(host='localhost', username='default', password='')

client.command("""
Drop Table idnet
""")

In [3]:
client.command("""
create table IF NOT EXISTS idnet10k_no_dupe
(
    name             text,
    imageData      text,
    prelabeled_fraud integer,
    id               text,
    person_name      text,
    address          text,
    birthday         text,
    gender           text,
    ethnicity        Nullable(text),
    class            text,
    issue_date       text,
    expire_date      text,
    eye_color        text,
    hair_color       Nullable(text),
    is_donor         text,
    is_veteran       text,
    license_number   text,
    height_inches    integer,
    weight_lbs       integer
)
    ENGINE = MergeTree
ORDER BY name
SETTINGS allow_nullable_key = 1
""")

In [4]:
#client.command()

In [5]:
df = pd.read_parquet('../silver/idnet10k_no_dupe.parquet')

In [6]:
print(df.columns)

Index(['address', 'birthday', 'class', 'ethnicity', 'expire_date', 'eye_color',
       'gender', 'hair_color', 'height_inches', 'id', 'imageData', 'is_donor',
       'is_veteran', 'issue_date', 'license_number', 'name', 'person_name',
       'prelabeled_fraud', 'weight_lbs'],
      dtype='object')


In [7]:
client.insert_df('idnet10k_no_dupe', df)

In [8]:
client.command('select * from idnet10k_no_dupe limit 1')

['az-generated.photos_v3_0006207.png',
 '...',
 '0',
 'az-generated.photos_v3_0006207.png',
 'Raul Campos',
 '25 Rodeo Road, Casa Grande, AZ 85122',
 '02/13/1991',
 'M',
 'latino',
 'C',
 '12/13/2021',
 '12/13/2026',
 'BRO',
 'BRO',
 'true',
 'false',
 'D78133264',
 '66',
 '219']

In [ ]:
#stop here for data load

In [25]:
client.command('SYSTEM RELOAD FUNCTIONS')

In [27]:
client.command('select test_function_python(2.0)')

'Value 2'

In [13]:
client.command('select test_function_python_venv(2.0)')

DatabaseError: Received ClickHouse exception, code: 46, server response: Code: 46. DB::Exception: Function with name `test_function_python_venv` does not exist. In scope SELECT test_function_python_venv(2.). Maybe you meant: ['test_function_python','test2_function_python']. (UNKNOWN_FUNCTION) (version 25.8.4.13 (official build)) (for url http://localhost:8123)

In [5]:
client.command("select torch_function_python(name) from idnet limit 1;")

DatabaseError: Received ClickHouse exception, code: 754, server response: Code: 754. DB::Exception: User defined function 'torch_function_python' failed. Command: 'clickhousetorch.py', Arguments: []. Original error: Code: 302. DB::Exception: Child process was exited with return code 1. (CHILD_WAS_NOT_EXITED_NORMALLY),. (UDF_EXECUTION_FAILED) (version 25.8.4.13 (official build)) (for url http://localhost:8123)

In [7]:
client.command('select * from idnet limit 1')

['generated.photos_v3_0000170',
 '...\\ncY/jMT4DwP8LiTbVZ1W9ua4AAAAASUVORK5CYII=',
 '0',
 'generated.photos_v3_0000170',
 'BRIAN WADE',
 '1436 Deer Valley Road, Peoria, AZ 85345',
 '1991-06-05',
 'M',
 'white',
 'D',
 '2022-10-09',
 '2027-10-09',
 'BRO',
 'BRO',
 'FALSE',
 'FALSE',
 'D82989586',
 '69',
 '118']

In [28]:
result = client.query('select name, xgboost_function_python(toString(gender), toString(ethnicity), toString(class), toString(eye_color), toString(hair_color), toString(is_donor), toString(is_veteran), toString(height_inches), toString(weight_lbs)) from idnet').result_rows
df = pd.DataFrame(result)
df

DatabaseError: Received ClickHouse exception, code: 349, server response: Code: 349. DB::Exception: Cannot convert NULL value to non-Nullable type: while executing 'FUNCTION xgboost_function_python(toString(__table1.gender) :: 10, toString(__table1.ethnicity) :: 0, toString(__table1.class) :: 1, toString(__table1.eye_color) :: 2, toString(__table1.hair_color) :: 3, toString(__table1.is_donor) :: 4, toString(__table1.is_veteran) :: 5, toString(__table1.height_inches) :: 6, toString(__table1.weight_lbs) :: 7) -> xgboost_function_python(toString(__table1.gender), toString(__table1.ethnicity), toString(__table1.c... Float32 : 8'. (CANNOT_INSERT_NULL_IN_ORDINARY_COLUMN) (version 25.8.4.13 (official build)) (for url http://localhost:8123)

In [ ]:
client.query('select test2_function_python(toString(gender), toString(ethnicity), toString(class), toString(eye_color), toString(hair_color), toString(is_donor), toString(is_veteran), toString(height_inches), toString(weight_lbs)) from idnet limit 5')


In [ ]:
client.query("select xgboost_function_python('M', 'white', 'D', 'BRO', 'BRO', 'FALSE', 'FALSE', '60', '120')")

In [ ]:
results = client.query("""
                       with cte as (
                           select name, torch_function_python(toString(name)) as pred from idnet
                           )
                       select * from cte where pred > 0.5
                       """).result_rows

In [ ]:
client.command('select * from idnet limit 1')

In [ ]:
for x in results:
    print(x)

In [ ]:
result = client.query("select gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran , height_inches, weight_lbs from idnet").result_rows
df = pd.DataFrame(result)
df

In [ ]:
result = client.query("select test2_function_python('M', 'white', 'D', 'BRO', 'BRO', 'FALSE', 'FALSE', '60', '120')").result_rows

print(result)

In [ ]:
result = client.query("select test2_function_python(gender, ethnicity, class, eye_color, hair_color, IF(is_donor == 'TRUE', 1,0) as is_donor, IF(is_veteran == 'TRUE', 1,0) as is_veteran, height_inches, weight_lbs) from idnet").result_rows
df = pd.DataFrame(result)
df


In [ ]:
result = client.query('select name, test2_function_python(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) from idnet').result_rows
df = pd.DataFrame(result)
df